# **Forecast Generation (30/90/365-day forecast)**

**Selecting a model for the forecast**  
Based on the model comparison, XGBoost (calendar-only) was selected as the primary forecasting model, having produced the lowest RMSE for both targets under fair, forecast-ready conditions. Prophet is included alongside it on the dashboard, both to provide uncertainty intervals (which XGBoost does not produce) and to illustrate an alternative assumption about future demand growth. SARIMA was excluded from this notebook as it was the slowest and least stable model to run, and it did not outperform either XGBoost or Prophet in the model comparison, so it was not carried forward into the forecast.

**Importing and loading data**  
Both models' libraries are imported here, along with pandas and numpy. The cleaned train and test sets built in the feature engineering notebook are loaded back in, since this notebook needs the full historical dataset rather than the raw session-level data.

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from prophet import Prophet

split_path = '/kaggle/input/datasets/sanyazuberi/ev-charging-train-test-split'

train = pd.read_csv(f'{split_path}/train.csv', index_col=0, parse_dates=True)
test = pd.read_csv(f'{split_path}/test.csv', index_col=0, parse_dates=True)

**Preparing the data**  
Since a forecast needs to begin immediately after the last observed date (Dec 31, 2024), train and test were recombined into one full historical dataset (full) before refitting. Using only the 90-day test set would have discarded three months of real, known data and started the forecast earlier than necessary. The same calendar-feature logic used in feature engineering (hour, day of week, month, weekend flag, year) is reapplied here, both to the historical data and to the newly generated future data range spanning 365 days beyond the last observed hour (8,760 hourly rows). Generating the full 365-day horizon in one pass, rather than three separate 30/90/365-day forecasts, means the 30- and 90-day views can simply be filtered from the same output later, via the days_ahead column.

In [2]:
full = pd.concat([train, test]).sort_index()
full.index.name = 'start_timestamp'

def add_calendar(df):
    df = df.copy()
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = df['day_of_week'].isin([5,6]).astype(int)
    df['year'] = df.index.year
    return df

calendar_features = ['hour', 'day_of_week', 'month', 'is_weekend', 'year']
targets = ['session_count', 'total_energy_kwh']

**Setting up the results list**  
An empty list (forecast) is created to collect each model's output as it's generated. Both the XGBoost and Prophet forecasting steps append their results to this same list, which is then combined into one table in a later step.

In [3]:
# Future timeline: 365 days of hourly rows after the last observed hour
horizon_hours = 365 * 24
future_index = pd.date_range(
    start=full.index.max() + pd.Timedelta(hours=1),
    periods=horizon_hours, freq = 'h'
)

future_cal = add_calendar(pd.DataFrame(index=future_index))
full_cal = add_calendar(full)

forecasts = []

**XGBoost forecast**  
The XGBoost model (calendar-only feature set) is refit on the entire historical dataset, not just the original training split, since the forecast should benefit from all available real data, including what was previously held out as the test set. It's then used to predict across the full 365-day future timeline.

In [4]:
# XGBoost (calendar only), refit on all data
for target in targets:
    model = XGBRegressor (n_estimators=200, max_depth=5, learning_rate=0.05, random_state = 42)
    model.fit(full_cal[calendar_features], full_cal[target])
    pred = model.predict(future_cal[calendar_features])
    forecasts.append(pd.DataFrame({
        'ds': future_index, 'model': 'XGBoost (calendar-only)', 'target': target,
        'yhat': pred, 'yhat_lower': np.nan, 'yhat_upper': np.nan
    }))
    

**Prophet forecast**  
Prophet is refit the same way, on the full historical dataset, using the multiplicative seasonality setting established earlier as the better-performing configuration. Unlike XGBoost, Prophet produces yhat_lower and yhat_upper uncertainty bounds, which are retained here for use in the dashboard.

In [5]:
for target in targets:
    p_train = full.reset_index()[['start_timestamp', target]]
    p_train = p_train.rename(columns={'start_timestamp': 'ds', target: 'y'})
    m = Prophet(seasonality_mode = 'multiplicative')
    m.fit(p_train)
    fut = m.make_future_dataframe(periods = horizon_hours, freq = 'h')
    fc = m.predict(fut).tail(horizon_hours)
    forecasts.append(pd.DataFrame({
        'ds': fc['ds'].values, 'model': 'Prophet', 'target': target,
        'yhat': fc['yhat'].values,
        'yhat_lower': fc['yhat_lower'].values, 'yhat_upper': fc['yhat_upper'].values     
    }))

15:46:42 - cmdstanpy - INFO - Chain [1] start processing
15:47:14 - cmdstanpy - INFO - Chain [1] done processing
15:47:23 - cmdstanpy - INFO - Chain [1] start processing
15:47:52 - cmdstanpy - INFO - Chain [1] done processing


**Combining, clipping and saving**  
Both models' forecasts are combined into a single table. Since session counts and energy usage cannot logically be negative, but Prophet's multiplicative seasonality can occasionally produce negative troughs (a limitation noted earlier), all predicted values, including the upper and lower bounds, are clipped to a minimum of zero before saving. A days_ahead column is also added at this stage, counting forward from the first forecasted day, so the single 365-day table can be filtered to the 30-day or 90-day view without needing to regenerate the forecast.

In [6]:
forecast_all = pd.concat(forecasts, ignore_index=True)

# Demand can't be negative
for col in ['yhat', 'yhat_lower', 'yhat_upper']:
    forecast_all[col] = forecast_all[col].clip(lower=0)

# days_ahead lets Power BI filter to 30/90/365-day horizons
start_day = future_index[0].normalize()
forecast_all['days_ahead'] = (forecast_all['ds'].dt.normalize() - start_day).dt.days +1

forecast_all.to_csv('/kaggle/working/forecast_all.csv', index = False)
forecast_all.head()

,ds,model,target,yhat,yhat_lower,yhat_upper,days_ahead
0,2025-01-01 00:00:00,XGBoost (calendar-only),session_count,4.576897,NaN,NaN,1
1,2025-01-01 01:00:00,XGBoost (calendar-only),session_count,4.603033,NaN,NaN,1
2,2025-01-01 02:00:00,XGBoost (calendar-only),session_count,4.624036,NaN,NaN,1
3,2025-01-01 03:00:00,XGBoost (calendar-only),session_count,4.272316,NaN,NaN,1
4,2025-01-01 04:00:00,XGBoost (calendar-only),session_count,4.424040,NaN,NaN,1


**Sanity check**  
Before treating the forecast as reliable, its output was checked against the most recent full year of actual data.

In [7]:
print(full[full.index.year == 2024][['session_count', 'total_energy_kwh']].mean())
print(forecast_all.shape)
print(forecast_all[forecast_all['model'] == 'Prophet'].head())
print(forecast_all.groupby(['model', 'target']).size())

session_count        19.040528
total_energy_kwh    882.034756
dtype: float64
(35040, 7)
                       ds    model         target      yhat  yhat_lower  \
17520 2025-01-01 00:00:00  Prophet  session_count  6.251351    1.663810   
17521 2025-01-01 01:00:00  Prophet  session_count  6.787480    2.208360   
17522 2025-01-01 02:00:00  Prophet  session_count  7.235606    2.257875   
17523 2025-01-01 03:00:00  Prophet  session_count  7.314936    2.545620   
17524 2025-01-01 04:00:00  Prophet  session_count  7.733333    3.057598   

       yhat_upper  days_ahead  
17520   10.882489           1  
17521   11.494655           1  
17522   11.870404           1  
17523   11.676283           1  
17524   12.219905           1  
model                    target          
Prophet                  session_count       8760
                         total_energy_kwh    8760
XGBoost (calendar-only)  session_count       8760
                         total_energy_kwh    8760
dtype: int64


This confirms four things: 
* the 2024 actual averages to compare forecasts against
* the table has the expected row count (8,760 hours * 2 models * 2 targets = 35,040 rows, with no duplicated)
* the Prophet's uncertainty bounds are populated correctly, since the default preview only shows XGBoost's rows
* each model/target combination has the full, complete set of hourly rows

**Result**  
2024 actual averages were 19.04 sessions/hour and 882.03 kWh/hour. XGBoost's forecast average matched closely (1903 sessions, 881.79 kWh), consistent with its inability to extrapolate growth beyond the years it was trained on. It holds essentially flat at the 2024 level. Prophet's forecast average was roughly 6% higher (20.28 sessions, 930.48 kWh), reflecting its continued extrapolation of the historical growth trend.  
This is a genuine, stated limitation rather than a discrepancy to resolve. The two models make different, unverifiable assumptions about whether the charging network's growth continued into 2025. XGBoost assumes it plateaus at 2024 levels. Prophet assumes it continues on trend. Both are included on the dashboard rather than presenting either as the single correct answer, so the person using the forecast can weigh the two scenarios rather than be given a false sense of certainty.